# Conversational Study Buddy Bot (LangChain)

This notebook demonstrates the use of LangChain Model I/O, LCEL Chains, Output Parsers, and Memory to build a simple conversational Study Buddy bot.

## Setup & Imports

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.memory import ConversationBufferMemory

# Ensure GOOGLE_API_KEY is set in environment

## Part 1: Basic Explainer Chain

In [ ]:
llm = ChatGoogleGenerativeAI(model='gemini-pro', temperature=0.3)

In [ ]:
prompt = ChatPromptTemplate.from_template(
    'Explain {topic} in one sentence as if to a beginner.'
)

chain = prompt | llm
response = chain.invoke({'topic': 'Quantum Computing'})
print(response.content)

## Part 2: Structured Output with Parser

In [ ]:
parser = CommaSeparatedListOutputParser()

structured_prompt = PromptTemplate(
    template=(
        'Explain {topic} simply and list three key points.\n'
        '{format_instructions}'
    ),
    input_variables=['topic'],
    partial_variables={'format_instructions': parser.get_format_instructions()}
)

structured_chain = structured_prompt | llm | parser
result = structured_chain.invoke({'topic': 'Machine Learning'})
print(result)

## Part 3: Conversational Memory

In [ ]:
memory = ConversationBufferMemory(memory_key='conversation_history', return_messages=True)

conversation_prompt = ChatPromptTemplate.from_template(
    'Conversation so far:\n{conversation_history}\nUser: {input}\nAI:'
)

conversation_chain = conversation_prompt | llm

In [ ]:
# Turn 1: User introduces name
user_input = 'Hi, my name is Rajesh.'
response = conversation_chain.invoke({
    'input': user_input,
    'conversation_history': memory.load_memory_variables({})['conversation_history']
})
print(response.content)
memory.save_context({'input': user_input}, {'output': response.content})

In [ ]:
# Turn 2: Ask if name is remembered
user_input = 'Do you remember my name?'
response = conversation_chain.invoke({
    'input': user_input,
    'conversation_history': memory.load_memory_variables({})['conversation_history']
})
print(response.content)
memory.save_context({'input': user_input}, {'output': response.content})

## Personal Reflection
This exercise helped reinforce my understanding of how LangChain chains, output parsers, and memory work together.
By incrementally building functionality, I learned how conversational context can be preserved without using RAG.